# DD-PRiSM-plus — Compare the experiments

**CPU session** — this only reads saved results, and should cost no GPU.

**Attach every experiment's output:** right panel → **Add Input → Your Work
→ Notebook Output**, once per notebook you want in the table:

| notebook | representation |
|---|---|
| `03_train` | `morgan` — the paper |
| `04_experiment_fusion` | `morgan+chemberta` |
| `05_experiment_chemberta` | `chemberta` |

Whichever are attached will appear. Missing ones are simply left out.

In [ ]:
import glob, json, os

found = sorted(glob.glob('/kaggle/input/**/runs*/evaluation.json',
                         recursive=True))
results, configs = {}, {}
for path in found:
    directory = os.path.dirname(path)
    label = os.path.basename(directory)
    label = 'morgan' if label == 'runs' else label.replace('runs-', '')
    results[label] = json.load(open(path))
    config = os.path.join(directory, 'config.json')
    if os.path.exists(config):
        configs[label] = json.load(open(config))

if not results:
    raise SystemExit('no evaluation.json found -- attach the experiment '
                     'notebook outputs via Add Input.')

# Baseline first; everything else is read as a change from it.
results = dict(sorted(results.items(), key=lambda kv: kv[0] != 'morgan'))

print(f"{'experiment':<20}{'drug input':>12}{'warm start':>12}")
for label in results:
    config = configs.get(label, {})
    started = 'yes' if config.get('init_from') else 'no'
    print(f"{label:<20}{config.get('drug_dim', '?'):>12}{started:>12}")

## Held-out results, side by side

**`unseen_drug` is the column this whole line of work is about.** The paper
names the limitation itself — *"we need more informative drug features for
the phenotypic prediction"* — and the baseline scores PCC 0.7585 there
against 0.9386 on unseen pairs.

`unseen_cellline` and `unseen_pair` are not expected to move much. They are
already above 0.93 and are not limited by how the drug is described.

In [ ]:
STAGES = ('pretrain', 'finetune', 'combination')
labels = list(results)

for stage in STAGES:
    rows = {k: v[stage] for k, v in results.items() if stage in v}
    if not rows:
        continue
    splits = sorted({s for r in rows.values() for s in r})
    print(f'\n{stage}')
    print(f"  {'split':<18}" + ''.join(f'{k:>24}' for k in rows))
    print(f"  {'':<18}" + ''.join(f"{'RMSE':>12}{'PCC':>12}" for _ in rows))
    for split in splits:
        line = f'  {split:<18}'
        for label in rows:
            got = rows[label].get(split)
            line += (f"{got['rmse']:>12.4f}{got['pcc']:>12.4f}" if got
                     else ' ' * 24)
        print(line)

## Did it actually help?

Change relative to the Morgan baseline, on every split. A negative RMSE
delta and a positive PCC delta are improvements.

In [ ]:
base = results.get('morgan')
if base is None:
    print('no morgan baseline attached -- nothing to compare against')
else:
    for label, values in results.items():
        if label == 'morgan':
            continue
        print(f'\n{label}  vs  morgan')
        for stage in STAGES:
            if stage not in values or stage not in base:
                continue
            for split in sorted(values[stage]):
                if split not in base[stage]:
                    continue
                got, ref = values[stage][split], base[stage][split]
                d_rmse = got['rmse'] - ref['rmse']
                d_pcc = got['pcc'] - ref['pcc']
                mark = '  <-- better' if (d_rmse < 0 and d_pcc > 0) else ''
                print(f'  {stage:<12}{split:<18}'
                      f'RMSE {d_rmse:+.4f}   PCC {d_pcc:+.4f}{mark}')

## A caution on reading this

One run per representation is one sample. A difference of a few thousandths
of RMSE is inside the noise of a different random seed and should not be
reported as an improvement. What would be a real result is `unseen_drug`
moving by a visible margin — its gap to `unseen_pair` is currently about
**0.18 PCC**, which is far larger than seed noise.

If the margin turns out to be small, the honest report is that a pretrained
embedding did not help here, which is a finding rather than a failure.